# a

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, classification_report

In [57]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("customer_dataset.csv")
print("Dataset loaded. Shape:", df.shape)
print("Churn rate:", "{:.2%}".format(df['Churn'].mean()))

# Features and target
X = df.drop(columns=["CustomerID", "Churn"])
y = df["Churn"]

# Split into training and testing sets with stratification (better for imbalanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train set size:", X_train.shape[0], "samples")
print("Test set size:", X_test.shape[0], "samples")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling complete.")

# Show dataset info
df.info()

# Display first few rows
df.head()


Dataset loaded. Shape: (100, 6)
Churn rate: 26.00%
Train set size: 80 samples
Test set size: 20 samples
Feature scaling complete.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          100 non-null    object 
 1   Tenure              100 non-null    int64  
 2   CallMinutes         100 non-null    int64  
 3   InternetUsage (GB)  100 non-null    float64
 4   Complaints          100 non-null    int64  
 5   Churn               100 non-null    int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 4.8+ KB


,CustomerID,Tenure,CallMinutes,InternetUsage (GB),Complaints,Churn
0,C001,12,300,15.2,0,0
1,C002,4,120,3.5,1,1
2,C003,24,600,40.1,0,0
3,C004,3,100,2.1,1,1
4,C005,18,450,20.6,0,0


In [4]:
df

,CustomerID,Tenure,CallMinutes,InternetUsage (GB),Complaints,Churn
0,C001,12,300,15.2,0,0
1,C002,4,120,3.5,1,1
2,C003,24,600,40.1,0,0
3,C004,3,100,2.1,1,1
4,C005,18,450,20.6,0,0
...,...,...,...,...,...,...
95,C096,26,669,55.1,4,0
96,C097,44,605,69.3,0,0
97,C098,34,389,65.4,4,0
98,C099,10,380,22.8,3,1


# b 

In [37]:
from sklearn.linear_model import LogisticRegression

# Build logistic regression model with balanced class weights and sufficient iterations
model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)

# Train the model on scaled training data
model.fit(X_train_scaled, y_train)
# Evaluate on test data
y_pred = model.predict(X_test_scaled)
from sklearn.metrics import precision_score
print("Precision:", precision_score(y_test, y_pred))
model

Precision: 0.125


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# c 

In [39]:
from sklearn.metrics import accuracy_score, precision_score, classification_report

# Predict the test set results
y_pred = model.predict(X_test_scaled)

# Calculate accuracy and precision
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)

# Print the results
print("Model Evaluation Results:")
print("Accuracy Score:", round(accuracy, 2))
print("Precision Score:", round(precision, 2))

print("\nDetailed Performance Metrics:")
print("Accuracy:", round(accuracy, 4), "(", round(accuracy * 100, 1), "%)")
print("Precision:", round(precision, 4), "(", round(precision * 100, 1), "%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Model Evaluation Results:
Accuracy Score: 0.45
Precision Score: 0.12

Detailed Performance Metrics:
Accuracy: 0.45 ( 45.0 %)
Precision: 0.125 ( 12.5 %)

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.53      0.59        15
           1       0.12      0.20      0.15         5

    accuracy                           0.45        20
   macro avg       0.40      0.37      0.37        20
weighted avg       0.53      0.45      0.48        20



## Dataframe style output for better readability 

In [59]:
# Summary metrics
metrics = {
    'Metric': ['Accuracy', 'Precision'],
    'Score': [accuracy, precision]
}
metrics_df = pd.DataFrame(metrics)
metrics_df['Score'] = metrics_df['Score'].round(4)


# Classification report as dict
report_dict = classification_report(y_test, y_pred, output_dict=True)

# Convert to DataFrame and round values
report_df = pd.DataFrame(report_dict).transpose().round(4)

print("\nClassification Report:")
report_df


Summary Metrics:

Classification Report:


,precision,recall,f1-score,support
0,0.6667,0.5333,0.5926,15.00
1,0.1250,0.2000,0.1538,5.00
accuracy,0.4500,0.4500,0.4500,0.45
macro avg,0.3958,0.3667,0.3732,20.00
weighted avg,0.5312,0.4500,0.4829,20.00


In [61]:
print("Summary Metrics:")
metrics_df

Summary Metrics:


,Metric,Score
0,Accuracy,0.450
1,Precision,0.125


### Why Precision is Important
- False positives = unnecessary retention costs
- High precision = efficient resource allocation
- Reduces wasted marketing spend on loyal customers

# d 

- We  can use something like SMOTE, which creates more examples of the "churn" class if it's way smaller than the "no churn" class. That way, the model won’t just focus on the bigger group and ignore the important one.

- Another way is to tune the model’s settings using tools like GridSearchCV. This helps you find the best combo of values (like the regularization strength or solver) that actually give better precision — especially useful if the default setup isn't working well.

# e 

Logistic Regression is simple and good for quick models, but it assumes a straight-line relationship between the features and the output — which doesn’t always make sense for real-life problems like churn. A better choice could be something like Random Forest or XGBoost, since they’re more flexible and can handle patterns that aren’t just linear. They usually perform better with precision too.

